In [110]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [111]:
%load_ext autoreload
%autoreload 2
# cell 1 — set up imports & paths
import sys
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

GNNGMM_DIR = PROJECT_ROOT / "scripts" / "GNNGMMAnalyzer"
if str(GNNGMM_DIR) not in sys.path:
    sys.path.insert(0, str(GNNGMM_DIR))

BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GNNGMM_DIR on sys.path:", GNNGMM_DIR.exists())
print("BACKEND_DIR on sys.path:", BACKEND_DIR.exists())

from embedding_pipeline_core import load_embeddings_from_files
from embedding_pipeline import EmbeddingPipeline

print("Modules imported OK")

# cell 2 — load embeddings and run pipeline
import torch   
from pathlib import Path

# Embeddings path
EMB_DIR = Path("data")

print("Embeddings dir exists:", EMB_DIR.exists())
if not EMB_DIR.exists():
    raise FileNotFoundError(f"Embeddings dir not found: {EMB_DIR}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PROJECT_ROOT: /home/frosty/Programs/Python/Latent_Space_Vis
GNNGMM_DIR on sys.path: True
BACKEND_DIR on sys.path: False
Modules imported OK
Embeddings dir exists: True


In [112]:
import torch
layer_files = ["emb_unique_cas.pt","output0_unique_cas.pt","output1_unique_cas.pt",
"output2_unique_cas.pt",
"output3_unique_cas.pt",
"output4_unique_cas.pt",
"mlp_input_unique_cas.pt",
"mlp_l1_unique_cas.pt",
"mlp_l2_unique_cas.pt"]
layer_names = [x.split('.')[0] for x in layer_files]#["emb_unique_cas","output0_unique_cas","output1_unique_cas"]
layer_files_dict = dict(zip(layer_names, layer_files))

In [113]:
import gudhi
from gudhi.dtm_rips_complex import DTMRipsComplex
from sklearn.preprocessing import StandardScaler
sts = []
for x in layer_files:
    d = StandardScaler().fit_transform(torch.load(f"{EMB_DIR}/{x}").squeeze().numpy())
    sc = DTMRipsComplex(points=d, k=1,q=2)
    st = sc.create_simplex_tree(max_dimension=1)
    st.collapse_edges()
    st.prune_above_filtration(3)
    st.compute_persistence()
    sts.append(st)

In [114]:
from latentlib.tracking import TDATracker, get_tracks_representatives
tracker = TDATracker(sts, 1)
tracks = get_tracks_representatives(tracker)

In [115]:
tracks.sort(key=lambda x: -len(x['representative_vertices']))

In [116]:
mapping = {i:0 for i in range(152)}
for x in tracks:
    for y in x['representative_vertices']:
        mapping[y] = x['track_id']
   # print(x)

In [123]:

# Call load_embeddings_from_files
# If your load_embeddings_from_files function supports passing only a directory, you can remove the second argument.
embeddings = load_embeddings_from_files(EMB_DIR, layer_files_dict)

print("Loaded layers:", list(embeddings.keys()))
for k, v in embeddings.items():
    try:
        print(k, getattr(v, "shape", (len(v),)))
    except Exception:
        print(k, type(v))

tmp_output_dir = PROJECT_ROOT / "frontend" / "public" / "data" / "tmp_notebook"
tmp_output_dir.mkdir(parents=True, exist_ok=True)

pipeline = EmbeddingPipeline(
    embeddings=embeddings,
    output_dir=tmp_output_dir,
    id_mapping=mapping,    
    n_components=4
)

print("Fitting GMMs...")
pipeline.fit_gmms()
print("Computing MDS / distance embedding...")
pipeline.compute_mds()

scatter_df = pipeline.generate_scatter_plot_data()
sankey_df = pipeline.generate_sankey_data()

print("scatter rows:", len(scatter_df))
print("sankey rows:", len(sankey_df))

display(scatter_df.head())
display(sankey_df.head())

  ✓ emb_unique_cas: shape (152, 128)
  ✓ output0_unique_cas: shape (152, 256)
  ✓ output1_unique_cas: shape (152, 256)
  ✓ output2_unique_cas: shape (152, 256)
  ✓ output3_unique_cas: shape (152, 256)
  ✓ output4_unique_cas: shape (152, 256)
  ✓ mlp_input_unique_cas: shape (152, 260)
  ✓ mlp_l1_unique_cas: shape (152, 130)
  ✓ mlp_l2_unique_cas: shape (152, 130)
Loaded layers: ['emb_unique_cas', 'output0_unique_cas', 'output1_unique_cas', 'output2_unique_cas', 'output3_unique_cas', 'output4_unique_cas', 'mlp_input_unique_cas', 'mlp_l1_unique_cas', 'mlp_l2_unique_cas']
emb_unique_cas (152, 128)
output0_unique_cas (152, 256)
output1_unique_cas (152, 256)
output2_unique_cas (152, 256)
output3_unique_cas (152, 256)
output4_unique_cas (152, 256)
mlp_input_unique_cas (152, 260)
mlp_l1_unique_cas (152, 130)
mlp_l2_unique_cas (152, 130)
Using 4 GMM components
Fitting GMMs...

Fitting GMMs for all layers...
  Fitting GMM for emb_unique_cas...
  Fitting GMM for mlp_input_unique_cas...
  Fitting 

,id,x,y,color_value,layer
0,0,1053.377761,1234.786399,0.0,emb_unique_cas
1,135,1557.396823,921.892890,0.0,emb_unique_cas
2,1,910.397481,-2696.177409,0.0,emb_unique_cas
3,35,3084.250371,813.262800,0.0,emb_unique_cas
4,36,1441.918031,-237.268487,0.0,emb_unique_cas


,real_id,from_layer,to_layer,from_cluster,to_cluster
0,0,emb_unique_cas,mlp_input_unique_cas,0,3
1,135,emb_unique_cas,mlp_input_unique_cas,0,3
2,1,emb_unique_cas,mlp_input_unique_cas,2,2
3,35,emb_unique_cas,mlp_input_unique_cas,1,3
4,36,emb_unique_cas,mlp_input_unique_cas,2,3


In [124]:
from latentvis import showSankey, showScatter

In [125]:
showSankey(sankey_df.to_dict(orient="records"))

In [129]:
showScatter(scatter_df.to_dict(orient="records"))